In [ ]:
# ==================================================
# 1. IMPORT & CONFIG
# ==================================================
import joblib
import numpy as np
import pandas as pd
import scipy.io as sio

from pathlib import Path
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
)

SEED = 42
np.random.seed(SEED)

# Safe bound for feature values to prevent overflow / extreme values
FEATURE_CLIP = 100_000.0
EPS = 1e-6

# Search for mill.mat in common locations
DATA_CANDIDATES = [
    Path("../data/mill.mat"),
    Path("data/mill.mat"),
    Path("mill.mat"),
]

mat_path = next((p for p in DATA_CANDIDATES if p.exists()), None)

if mat_path is None:
    raise FileNotFoundError(
        "mill.mat file not found. "
        "Make sure it is in the ../data/, data/, or notebook folder."
    )

print(f"--> Using file: {mat_path.resolve()}")

TARGET_INDEX = 2
SENSOR_INDICES = [7, 8, 9, 10, 11, 12]
CHANNEL_NAMES = [
    "smc",
    "smd",
    "vib_table",
    "vib_spindle",
    "ae_table",
    "ae_spindle",
]

SIGNAL_LENGTH = 9000
THRESHOLD_MM = 0.18
ALERT_PROB_THRESHOLD = 0.65

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ==================================================
# 2. HELPER FUNCTIONS
# ==================================================
def to_scalar(value):
    """
    Extract scalar value from MATLAB field.
    If failed, return NaN.
    """
    try:
        arr = np.asarray(value).squeeze()
        if arr.size == 0:
            return np.nan
        return float(arr.reshape(-1)[0])
    except Exception:
        return np.nan


def to_signal(value, length=SIGNAL_LENGTH):
    """
    Convert MATLAB field to 1D signal with fixed length.
    Longer -> truncate.
    Shorter -> zero padding.
    """
    try:
        sig = np.asarray(value, dtype=np.float64).squeeze().reshape(-1)
    except Exception:
        return None

    if sig.size == 0:
        return None

    if sig.size > length:
        sig = sig[:length]
    elif sig.size < length:
        sig = np.pad(
            sig,
            pad_width=(0, length - sig.size),
            mode="constant",
            constant_values=0.0,
        )

    if not np.all(np.isfinite(sig)):
        return None

    return sig


def safe_float(value):
    """
    Safely convert value to float.
    If NaN/Inf/error, return NaN.
    Extreme values are clipped to safe range.
    """
    try:
        value = float(value)
        if not np.isfinite(value):
            return np.nan
        return float(np.clip(value, -FEATURE_CLIP, FEATURE_CLIP))
    except Exception:
        return np.nan


def clean_features(X):
    """
    Clean infinity and clip extreme values.
    NaN is left for SimpleImputer to handle later.
    """
    if isinstance(X, pd.DataFrame):
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.apply(pd.to_numeric, errors="coerce")
        X = X.clip(lower=-FEATURE_CLIP, upper=FEATURE_CLIP)
        return X.astype(np.float64)

    X = np.asarray(X, dtype=np.float64)
    X[~np.isfinite(X)] = np.nan
    X = np.clip(X, -FEATURE_CLIP, FEATURE_CLIP)
    return X


# ==================================================
# 3. FEATURE EXTRACTION
# ==================================================
CHANNEL_STATS = [
    "mean",
    "std",
    "rms",
    "abs_mean",
    "peak",
    "p2p",
    "crest",
    "energy",
    "skew",
    "kurt",
    "zero_cross",
]


def compute_channel_features(sig, prefix):
    """
    Extract statistical features from a single signal channel.
    """
    sig = np.asarray(sig, dtype=np.float64).reshape(-1)
    sig = sig[np.isfinite(sig)]

    feats = {}

    if len(sig) < 10:
        for stat in CHANNEL_STATS:
            feats[f"{prefix}_{stat}"] = np.nan
        return feats

    # Clip signal first to prevent statistics from exploding
    sig = np.clip(sig, -FEATURE_CLIP, FEATURE_CLIP)

    mean = float(np.mean(sig))
    std = float(np.std(sig))
    rms = float(np.sqrt(np.mean(sig**2)))
    abs_mean = float(np.mean(np.abs(sig)))
    peak = float(np.max(np.abs(sig)))
    p2p = float(np.max(sig) - np.min(sig))
    crest = peak / rms if rms > EPS else 0.0
    energy = float(np.mean(sig**2))

    if std > EPS:
        z = (sig - mean) / std
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4) - 3.0)
    else:
        skew = 0.0
        kurt = 0.0

    zero_cross = float(np.sum(np.diff(np.sign(sig - mean)) != 0))

    values = [
        mean,
        std,
        rms,
        abs_mean,
        peak,
        p2p,
        crest,
        energy,
        skew,
        kurt,
        zero_cross,
    ]

    for stat, val in zip(CHANNEL_STATS, values):
        feats[f"{prefix}_{stat}"] = safe_float(val)

    return feats


def extract_features_from_signals(signals, metadata=None):
    """
    Inference function to transform 6 signal channels into a single feature row.

    signals: list/array containing 6 signals with length 9000
    metadata: optional dict, example:
        {
            "DOC": 1.0,
            "Feed": 0.5,
            "Material": 0.0
        }
    """
    if len(signals) != len(CHANNEL_NAMES):
        raise ValueError(
            f"Number of signals must be {len(CHANNEL_NAMES)}, "
            f"but got {len(signals)}"
        )

    feats = {}

    for ch, sig in zip(CHANNEL_NAMES, signals):
        feats.update(compute_channel_features(sig, ch))

    metadata = metadata or {}
    feats["DOC"] = safe_float(metadata.get("DOC", np.nan))
    feats["Feed"] = safe_float(metadata.get("Feed", np.nan))
    feats["Material"] = safe_float(metadata.get("Material", np.nan))

    return feats


# ==================================================
# 4. LOAD DATASET + FEATURE EXTRACTION
# ==================================================
mat_file = sio.loadmat(str(mat_path))

# Handle potential key names with spaces
mat_key_map = {k.strip(): k for k in mat_file.keys()}

if "mill" not in mat_key_map:
    raise KeyError(
        f"Field 'mill' not found. Available keys: {list(mat_file.keys())}"
    )

runs = mat_file[mat_key_map["mill"]][0]

print("--> Extracting features from 6 sensor channels...")

records = []
targets = []
groups = []
times = []
skipped = 0

for i, run in enumerate(runs):
    try:
        vb = safe_float(to_scalar(run[TARGET_INDEX]))

        if not np.isfinite(vb) or vb < 0:
            skipped += 1
            continue

        signals = []
        valid = True

        for idx in SENSOR_INDICES:
            sig = to_signal(run[idx])
            if sig is None:
                valid = False
                break
            signals.append(sig)

        if not valid:
            skipped += 1
            continue

        # Signal feature extraction
        feats = extract_features_from_signals(signals)

        # Additional metadata, but Case/Run are not used as features
        case_id = to_scalar(run[0])
        run_id = to_scalar(run[1])
        time_val = to_scalar(run[3])
        doc = to_scalar(run[4])
        feed = to_scalar(run[5])
        material = to_scalar(run[6])

        feats["DOC"] = safe_float(doc)
        feats["Feed"] = safe_float(feed)
        feats["Material"] = safe_float(material)

        records.append(feats)
        targets.append(vb)

        case_txt = str(int(case_id)) if np.isfinite(case_id) else "NA"
        run_txt = str(int(run_id)) if np.isfinite(run_id) else str(i)
        groups.append(f"{case_txt}_{run_txt}")

        times.append(safe_float(time_val))

    except Exception:
        skipped += 1
        continue

if len(records) == 0:
    raise ValueError("No valid samples were successfully extracted.")

X = pd.DataFrame(records)
y = np.asarray(targets, dtype=np.float64)
groups = np.asarray(groups)
times = np.asarray(times, dtype=np.float64)

# Sanity check target unit
if y.max() > 10:
    print(
        "[INFO] y.max() > 10. "
        "Likely unit is still in µm, automatically converting to mm: y = y / 1000."
    )
    y = y / 1000.0

# Clean features before analysis / split
X = clean_features(X)
X = X.dropna(axis=1, how="all")

if X.shape[1] == 0:
    raise ValueError("No valid features after cleaning.")

print("\n==========================================")
print("FEATURE EXTRACTION RESULTS")
print("==========================================")
print(f"Total valid samples    : {len(y)}")
print(f"Total skipped samples : {skipped}")
print(f"Initial feature count  : {X.shape[1]}")
print(f"y min                 : {y.min():.4f} mm")
print(f"y max                 : {y.max():.4f} mm")
print(f"y mean                : {y.mean():.4f} mm")
print(f"y std                 : {y.std():.4f} mm")
print(f"Critical samples > {THRESHOLD_MM} mm: {(y > THRESHOLD_MM).sum()}")


# ==================================================
# 5. SAFER DATA SPLIT
# ==================================================
def make_split(X, y, groups, threshold):
    """
    Priority:
    1. GroupShuffleSplit based on Case_Run
    2. Fallback to stratified random split if group split is unbalanced
    """
    y_crit = (y > threshold).astype(int)
    unique_groups = np.unique(groups)

    if len(unique_groups) >= 5:
        try:
            splitter = GroupShuffleSplit(
                n_splits=1,
                test_size=0.20,
                random_state=SEED
            )
            train_idx, test_idx = next(splitter.split(X, y, groups=groups))

            train_has_both_classes = len(np.unique(y[train_idx] > threshold)) > 1
            test_has_both_classes = len(np.unique(y[test_idx] > threshold)) > 1

            if train_has_both_classes and test_has_both_classes:
                print("[INFO] Using GroupShuffleSplit based on Case_Run.")
                return train_idx, test_idx

        except Exception:
            pass

    stratify = y_crit if len(np.unique(y_crit)) > 1 else None

    print("[INFO] Fallback to stratified/random train_test_split.")
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.20,
        random_state=SEED,
        stratify=stratify,
    )

    return train_idx, test_idx


train_idx, test_idx = make_split(X, y, groups, THRESHOLD_MM)

X_train = clean_features(X.iloc[train_idx])
X_test = clean_features(X.iloc[test_idx])

y_train = y[train_idx]
y_test = y[test_idx]

# Drop columns that are all NaN in train, then align test columns
cols_to_drop = X_train.columns[X_train.isna().all()].tolist()

if cols_to_drop:
    X_train = X_train.drop(columns=cols_to_drop)
    X_test = X_test.drop(columns=cols_to_drop)

feature_columns = X_train.columns.tolist()
X_test = X_test[feature_columns]

if len(feature_columns) == 0:
    raise ValueError("No features remaining after split and cleaning.")

# Final validation: no infinity allowed, NaN is ok since imputer will handle it
for split_name, df in [("train", X_train), ("test", X_test)]:
    arr = df.to_numpy(dtype=np.float64)
    if np.isinf(arr).any():
        raise ValueError(f"There is still infinity in X_{split_name}.")

print("\n==========================================")
print("DATA SPLIT")
print("==========================================")
print(f"Train samples : {len(X_train)}")
print(f"Test samples  : {len(X_test)}")
print(f"Features used  : {len(feature_columns)}")
print(f"Train critical : {(y_train > THRESHOLD_MM).sum()}")
print(f"Test critical  : {(y_test > THRESHOLD_MM).sum()}")


# ==================================================
# 6. TRAIN RANDOM FOREST REGRESSOR
# ==================================================
regressor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=700,
        max_depth=14,
        min_samples_split=5,
        min_samples_leaf=4,
        max_features=0.35,
        max_samples=0.8,
        bootstrap=True,
        oob_score=True,
        random_state=SEED,
        n_jobs=-1
    ))
])

print("\n--> Training RandomForestRegressor...")
regressor.fit(X_train, y_train)


# ==================================================
# 7. REGRESSION EVALUATION
# ==================================================
y_pred = regressor.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--> Model Evaluation on Test Data:")
print(f"    Test MAE : {mae:.4f} mm")
print(f"    Test RMSE: {rmse:.4f} mm")
print(f"    Test R2  : {r2:.4f}")


# ==================================================
# 8. CRISIS THRESHOLD EVALUATION
# ==================================================
y_true_critical = (y_test > THRESHOLD_MM).astype(int)
y_pred_critical = (y_pred > THRESHOLD_MM).astype(int)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true_critical,
    y_pred_critical,
    average="binary",
    zero_division=0
)

cm = confusion_matrix(y_true_critical, y_pred_critical, labels=[0, 1])

print("\n--> Crisis Threshold Evaluation (VB > 0.18 mm):")
print(f"    Precision : {precision:.4f}")
print(f"    Recall    : {recall:.4f}")
print(f"    F1-Score  : {f1:.4f}")
print("    Confusion Matrix [TN, FP / FN, TP]:")
print(cm)


# ==================================================
# 9. TRAIN RANDOM FOREST CLASSIFIER FOR PROBABILITY
# ==================================================
classifier = None

y_train_critical = (y_train > THRESHOLD_MM).astype(int)
y_test_critical = (y_test > THRESHOLD_MM).astype(int)

if len(np.unique(y_train_critical)) > 1:
    classifier = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf_cls", RandomForestClassifier(
            n_estimators=700,
            max_depth=14,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features="sqrt",
            max_samples=0.8,
            bootstrap=True,
            oob_score=True,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1
        ))
    ])

    print("\n--> Training RandomForestClassifier for damage probability...")
    classifier.fit(X_train, y_train_critical)

    y_prob = classifier.predict_proba(X_test)[:, 1]

    try:
        auc = roc_auc_score(y_test_critical, y_prob)
    except ValueError:
        auc = np.nan

    y_prob_alert = (y_prob >= ALERT_PROB_THRESHOLD).astype(int)

    precision_prob, recall_prob, f1_prob, _ = precision_recall_fscore_support(
        y_test_critical,
        y_prob_alert,
        average="binary",
        zero_division=0
    )

    cm_prob = confusion_matrix(y_test_critical, y_prob_alert, labels=[0, 1])

    print("\n--> Crisis Classifier Evaluation:")
    print(f"    AUC Score           : {auc:.4f}")
    print(f"    Precision @ {ALERT_PROB_THRESHOLD:.0%} : {precision_prob:.4f}")
    print(f"    Recall @ {ALERT_PROB_THRESHOLD:.0%}    : {recall_prob:.4f}")
    print(f"    F1-Score @ {ALERT_PROB_THRESHOLD:.0%}  : {f1_prob:.4f}")
    print("    Confusion Matrix [TN, FP / FN, TP]:")
    print(cm_prob)

else:
    auc = np.nan
    precision_prob = np.nan
    recall_prob = np.nan
    f1_prob = np.nan
    print("\n[WARNING] Training data only has 1 critical class.")
    print("Probability classifier not trained.")


# ==================================================
# 10. EXPORT MODEL FOR PRODUCTION
# ==================================================
training_metrics = {
    "regression": {
        "mae_mm": float(mae),
        "rmse_mm": float(rmse),
        "r2": float(r2),
    },
    "threshold_regression": {
        "threshold_mm": THRESHOLD_MM,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    },
}

if classifier is not None:
    training_metrics["classifier"] = {
        "auc": float(auc) if np.isfinite(auc) else None,
        "alert_prob_threshold": ALERT_PROB_THRESHOLD,
        "precision": float(precision_prob) if np.isfinite(precision_prob) else None,
        "recall": float(recall_prob) if np.isfinite(recall_prob) else None,
        "f1": float(f1_prob) if np.isfinite(f1_prob) else None,
    }

export_bundle = {
    "model_type": "RandomForestRegressor+RandomForestClassifier",
    "regressor": regressor,
    "classifier": classifier,
    "feature_columns": feature_columns,
    "channel_names": CHANNEL_NAMES,
    "signal_length": SIGNAL_LENGTH,
    "threshold_mm": THRESHOLD_MM,
    "alert_prob_threshold": ALERT_PROB_THRESHOLD,
    "target_unit": "mm",
    "target_stats": {
        "min": float(y.min()),
        "max": float(y.max()),
        "mean": float(y.mean()),
        "std": float(y.std()),
    },
    "training_metrics": training_metrics,
    "created_at": datetime.now().isoformat(),
    "version": "2.1-rf-feature-extraction-clean",
}

model_path = MODEL_DIR / "random_forest_model.pkl"
joblib.dump(export_bundle, model_path)

print(f"\n✅ Model successfully saved to: {model_path.resolve()}")

--> Menggunakan file: D:\MyProject\MLops\predictive-maintenance-pipeline\data\mill.mat
--> Mengekstraksi fitur dari 6 channel sensor...

HASIL FEATURE EXTRACTION
Total sampel valid   : 146
Total sampel skipped : 21
Jumlah fitur awal    : 69
y min                : 0.0000 mm
y max                : 1.5300 mm
y mean               : 0.3376 mm
y std                : 0.2596 mm
Sampel kritis > 0.18 mm: 98
[INFO] Menggunakan GroupShuffleSplit berdasarkan Case_Run.

DATA SPLIT
Train samples : 116
Test samples  : 30
Fitur dipakai : 69
Train kritis  : 77
Test kritis   : 21

--> Training RandomForestRegressor...

--> Evaluasi Model pada Data Uji:
    Test MAE : 0.1045 mm
    Test RMSE: 0.1558 mm
    Test R2  : 0.6789

--> Evaluasi Threshold Krisis (VB > 0.18 mm):
    Precision : 0.8750
    Recall    : 1.0000
    F1-Score  : 0.9333
    Confusion Matrix [TN, FP / FN, TP]:
[[ 6  3]
 [ 0 21]]

--> Training RandomForestClassifier untuk probabilitas kerusakan...

--> Evaluasi Classifier Krisis:
    AUC S